# Right Agent Group -- full stack + REAL CALL test on Kaggle GPU

**Two ways to use this notebook -- pick based on what you need:**

### A) Interactive (for a LIVE, clickable session -- use this to actually test the dashboard or place a call)
Open this notebook on kaggle.com yourself, in your own browser: Settings (right panel) -> Accelerator = **GPU T4 x2**, Internet = **ON**, then click **Run All** from the Kaggle UI.
Watch the output live -- when the "login cookie + URLs" cell prints, the container is still running, so open them immediately. The session stays alive until you close it or Kaggle's time limit hits -- no keep-alive cell needed.

TTS is self-hosted Edge TTS (free Microsoft neural voices, CPU-only -- no GPU, no API key, nothing to download): it is started and gate-tested with a real Telugu synthesis as part of this same Run All, BEFORE the website starts -- so a broken voice stops the run loudly instead of producing silent calls.

### B) API push (`kaggle kernels push -p kaggle/`) -- headless live session
Kaggle runs API-pushed notebooks in **batch/commit mode** and normally tears the container down the instant the last cell finishes. Two cells make this mode usable for a REAL live session anyway: the URLs cell pushes the tunnel URLs + login cookie to the `kaggle-live` branch of the GitHub repo (readable from your machine while the kernel is still running), and the final keep-alive cell sleeps for hours so the website stays up. Stop the kernel on kaggle.com when you're done (it burns GPU quota while alive).

⚠️ This notebook reads all secrets from **Kaggle Secrets** (Add-ons -> Secrets in the notebook editor) -- it never embeds real credentials in the file itself. Before running, add these secrets (skip any you don't use yet):
`GITHUB_TOKEN`, `PG_PASSWORD`, `GOOGLE_CLIENT_ID`, `GOOGLE_CLIENT_SECRET`, `AUTH_SECRET`, `GROQ_API_KEY`, `EXOTEL_SID`, `EXOTEL_API_KEY`, `EXOTEL_API_TOKEN`, `EXOTEL_CALLER_ID`, `EXOTEL_FLOW_APP_ID`, `WHATSAPP_SERVICE_KEY`, `WHATSAPP_TOKEN`, `WHATSAPP_PHONE_NUMBER_ID`, `WHATSAPP_APP_SECRET`.

In [ ]:
%%bash
# ---- 1. System dependencies: Node 22, PostgreSQL, ffmpeg, cloudflared ----
set -e
curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1
apt-get install -y -qq nodejs postgresql ffmpeg > /dev/null 2>&1
curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /usr/local/bin/cloudflared
echo "node: $(node -v) | psql: $(psql --version | awk '{print $3}') | ffmpeg + cloudflared OK"

In [ ]:
from kaggle_secrets import UserSecretsClient
import subprocess

# ---- 2. Clone the private repo + start PostgreSQL with matching password ----
secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
subprocess.run(
    ["git", "clone", "--quiet",
     f"https://{github_token}@github.com/arjungaming371-cmyk/right-agent-group.git",
     "/kaggle/working/app"],
    check=True,
)
print(subprocess.run(["git", "-C", "/kaggle/working/app", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

In [ ]:
# ---- 3. Write .env for this session only (not stored in the repo) ----
# All real secrets come from Kaggle Secrets (Add-ons -> Secrets), never from this file.
import io
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def secret(name, default=""):
    try:
        return secrets.get_secret(name)
    except Exception:
        return default


PG_PASSWORD = secret("PG_PASSWORD")
GOOGLE_CLIENT_ID = secret("GOOGLE_CLIENT_ID")
GOOGLE_CLIENT_SECRET = secret("GOOGLE_CLIENT_SECRET")
AUTH_SECRET = secret("AUTH_SECRET")
GROQ_API_KEY = secret("GROQ_API_KEY")  # optional — Groq brain; Ollama used if empty
EXOTEL_SID = secret("EXOTEL_SID", "aiagent25")
EXOTEL_API_KEY = secret("EXOTEL_API_KEY")
EXOTEL_API_TOKEN = secret("EXOTEL_API_TOKEN")
EXOTEL_CALLER_ID = secret("EXOTEL_CALLER_ID", "09513886363")
EXOTEL_FLOW_APP_ID = secret("EXOTEL_FLOW_APP_ID", "1291862")
WHATSAPP_SERVICE_KEY = secret("WHATSAPP_SERVICE_KEY")
WHATSAPP_TOKEN = secret("WHATSAPP_TOKEN")
WHATSAPP_PHONE_NUMBER_ID = secret("WHATSAPP_PHONE_NUMBER_ID")
WHATSAPP_APP_SECRET = secret("WHATSAPP_APP_SECRET")

ENV = f"""# ============================================================
# Right Agent Group FINAL_10 (Cloud API) — .env
# ❌ = must fill before starting   ⭕ = fill when going live
# ============================================================

# --- App URL (no trailing slash, must be https for OAuth + Meta webhook) ---
# LOCAL TESTING — switch to your https tunnel domain before going live
# (WhatsApp webhook + Exotel need https; Google login works on localhost)
NEXT_PUBLIC_APP_URL=http://localhost:3000

# --- PostgreSQL ---
PG_HOST=localhost
PG_PORT=5432
PG_DATABASE=right_agent_group
PG_USER=postgres
PG_PASSWORD={PG_PASSWORD}

# --- Priya's brain: Groq primary (if key present), Ollama fallback ---
GROQ_API_KEY={GROQ_API_KEY}
OLLAMA_URL=http://localhost:11434
OLLAMA_MODEL=llama3.1:8b
OLLAMA_GPU=false                                     # IdeaPad Slim 3 = no NVIDIA GPU
OLLAMA_MAX_CONCURRENT=1                              # CPU-only: keep at 1

# --- Google Login (console.cloud.google.com/apis/credentials) ---
# Redirect URI must be exactly: https://your-domain.com/api/auth/google/callback
GOOGLE_CLIENT_ID={GOOGLE_CLIENT_ID}
GOOGLE_CLIENT_SECRET={GOOGLE_CLIENT_SECRET}
AUTH_SECRET={AUTH_SECRET}
ADMIN_EMAIL=arjun996625@gmail.com

# --- Exotel ---
EXOTEL_SID={EXOTEL_SID}
EXOTEL_API_KEY={EXOTEL_API_KEY}
EXOTEL_API_TOKEN={EXOTEL_API_TOKEN}
EXOTEL_SUBDOMAIN=api.exotel.com
EXOTEL_CALLER_ID={EXOTEL_CALLER_ID}                         # ExoPhone 095-138-86363
EXOTEL_FLOW_APP_ID={EXOTEL_FLOW_APP_ID}                           # aiagent23 Landing Flow (App Bazaar)

# --- WhatsApp Business Cloud API (SETUP-GUIDE-CLOUD-API.md steps 3,5,6) ---
WHATSAPP_TOKEN={WHATSAPP_TOKEN}                      # from Kaggle Secrets — permanent System User token (EAA...)
WHATSAPP_PHONE_NUMBER_ID={WHATSAPP_PHONE_NUMBER_ID}  # from Kaggle Secrets — Phone Number ID (not the phone number)
WHATSAPP_VERIFY_TOKEN=rag-verify-2026                # ❌ any string — must match Meta webhook setup
WHATSAPP_APP_SECRET={WHATSAPP_APP_SECRET}            # from Kaggle Secrets — REQUIRED for webhook signature verification
WHATSAPP_FORM_TEMPLATE=loan_application_form
# Auto-sent once per call (create + get these approved in WhatsApp Manager too):
WHATSAPP_CALL_FOLLOWUP_TEMPLATE=call_followup           # sent after any completed call with real conversation
WHATSAPP_MISSED_CALL_TEMPLATE=missed_call_followup      # sent when a call is missed/busy/no-answer

# --- Internal service auth (one shared secret for app ↔ voicebot ↔ STT) ---
WHATSAPP_SERVICE_KEY={WHATSAPP_SERVICE_KEY}
STT_API_KEY={WHATSAPP_SERVICE_KEY}

# --- STT (Whisper) ---
STT_SERVICE_URL=http://127.0.0.1:3003
STT_FORCE_DEVICE=cpu                                 # explicit: no CUDA on this laptop
# small (~460MB) instead of large-v3 (~3GB): the large-v3 HF download has
# stalled for 30+ minutes on Kaggle (unauthenticated rate limits), leaving
# calls with no STT at all. small downloads in under a minute and on a T4
# GPU is fast and accurate enough for collecting name/city/number.
STT_MODEL=small

# --- TTS (Priya's voice) ---
# Self-hosted Edge TTS (server/tts-service) — free Microsoft neural voices,
# CPU-only, no GPU/API key, nothing to download. Started + gate-tested by
# the cells below. Dashboard "speak" uses Edge too (TTS_PROVIDER=edge default).
TTS_SERVICE_URL=http://127.0.0.1:3004

# --- Voicebot ---
VOICEBOT_PORT=3002
APP_INTERNAL_URL=http://127.0.0.1:3000
# Diagnostics: dump every captured caller utterance as a .wav so a silent-call
# problem can be inspected with one command. Harmless to leave on.
VOICEBOT_DEBUG_DIR=/kaggle/working
# STT: Whisper's own VAD stays OFF (voicebot already endpoints); it was silently
# dropping real 8kHz phone audio as non-speech. Do not set STT_VAD_FILTER here.
STT_VAD_FILTER=0

# --- Cloudflare NAMED tunnel (stable URL — required for OAuth/Meta/Exotel) ---
CF_TUNNEL_NAME=rag                                   # ❌ create: cloudflared tunnel create rag

# --- Email confirmations (optional — leave blank to skip) ---
SMTP_HOST=
SMTP_PORT=465
SMTP_USER=
SMTP_PASS=
SMTP_FROM="Right Agent Group <yourbusiness@gmail.com>"

# --- Optional overrides ---
# APPLICATION_FORM_URL=                              # only if the form lives elsewhere
"""
io.open("/kaggle/working/app/.env", "w", encoding="utf-8").write(ENV)
print("wrote .env,", len(ENV), "chars — secrets loaded from Kaggle Secrets")

In [ ]:
%%bash
# ---- 4. Ollama on GPU + pull the model (~5 GB, the slow step) ----
# NOTE: the official curl|sh installer assumes systemd, which Kaggle
# containers do not have, so it fails silently. Installing the binary
# directly and running "ollama serve" ourselves avoids that entirely.
# Ollama ships releases as .tar.zst (zstd), not .tgz, as of v0.31.x —
# fetched from the GitHub release directly since ollama.com/download
# 404s for this asset name.
set -e
apt-get install -y -qq zstd > /dev/null 2>&1
curl -L -o /tmp/ollama.tar.zst https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst
tar --zstd -C /usr -xf /tmp/ollama.tar.zst
nohup ollama serve > /kaggle/working/ollama.log 2>&1 &
sleep 8
ollama pull llama3.1:8b
echo "--- GPU check ---"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%bash
# ---- 5. Install app dependencies (root + voicebot + STT) ----
set -e
cd /kaggle/working/app
npm install --silent 2>&1 | tail -1
cd server && npm install --silent 2>&1 | tail -1 && cd ..
pip install -q -r server/stt-service/requirements.txt
pip install -q -r server/tts-service/requirements.txt
echo "All dependencies installed"

In [ ]:
%%bash
# ---- 6. Edge TTS: start (CPU — no GPU, no model download) + poll ----
# Free Microsoft neural voices via edge-tts. Replaces IndicF5 entirely:
# nothing to download, warm in seconds. GPUs stay free for Ollama + Whisper.
set -e
cd /kaggle/working/app/server/tts-service
nohup python -u -m uvicorn app:app --host 127.0.0.1 --port 3004 > /kaggle/working/tts.log 2>&1 &
echo "started, pid $!"
KEY=$(grep -E '^WHATSAPP_SERVICE_KEY=' /kaggle/working/app/.env | cut -d= -f2 | awk '{print $1}')
UP=0
for i in $(seq 1 15); do
  sleep 2
  if curl -sf -o /dev/null --max-time 3 -H "x-api-key: $KEY" http://127.0.0.1:3004/health 2>/dev/null; then
    echo "[$((i*2))s] TTS responding"; UP=1; break
  fi
  echo "[$((i*2))s] not up yet..."
done
if [ "$UP" != "1" ]; then
  echo "Edge TTS never came up — tts.log:"; cat /kaggle/working/tts.log; exit 1
fi
curl -s -H "x-api-key: $KEY" http://127.0.0.1:3004/health; echo ""


In [ ]:
%%bash
# ---- 7. Edge TTS: real synthesis gate (Telugu) ----
# This is the ONLY phone TTS provider (no fallback), so a failure here means
# every call would go silent. Fail loudly and stop Run All rather than start
# a broken website.
set -e
cd /kaggle/working/app
KEY=$(grep -E '^WHATSAPP_SERVICE_KEY=' .env | cut -d= -f2 | awk '{print $1}')
cat > /tmp/tts_test_body.json << 'JSONEOF'
{"text":"నమస్కారం! నేను ప్రియ. మీకు లోన్ గురించి సహాయం చేస్తాను.","language":"telugu"}
JSONEOF
echo "--- synthesizing (timing it) ---"
time curl -s -X POST http://127.0.0.1:3004/synthesize -H "Content-Type: application/json" -H "x-api-key: $KEY" -d @/tmp/tts_test_body.json -o /kaggle/working/tts-test.mp3
SIZE=$(stat -c%s /kaggle/working/tts-test.mp3 2>/dev/null || echo 0)
echo "--- result: ${SIZE} bytes (mp3) ---"
if [ "$SIZE" -lt 5000 ]; then
  echo "tts-test.mp3 is only ${SIZE} bytes -- looks like an error, not real audio."
  head -c 300 /kaggle/working/tts-test.mp3; echo ""
  echo "--- tts.log ---"; tail -20 /kaggle/working/tts.log
  exit 1
fi
echo "Edge TTS confirmed working -- Priya has a voice. Safe to continue."


In [ ]:
# ---- 8. TWO public tunnels: website (3000) + voicebot WebSocket (3002) ----
import subprocess, re

def quick_tunnel(port):
    p = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(90):
        line = p.stdout.readline()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m: return m.group(0)
    raise RuntimeError(f"tunnel for port {port} failed — re-run this cell")

web_url = quick_tunnel(3000)
vb_url  = quick_tunnel(3002)
wss_url = vb_url.replace("https://", "wss://") + "/voicebot"

open("/kaggle/working/tunnel_url.txt", "w").write(web_url)
open("/kaggle/working/wss_url.txt", "w").write(wss_url)
print("WEBSITE URL :", web_url)
print("VOICEBOT WSS:", wss_url)

In [ ]:
%%bash
# ---- 9. Point .env at the tunnel + GPU settings, then create the database ----
set -e
cd /kaggle/working/app
URL=$(cat /kaggle/working/tunnel_url.txt)
sed -i "s|^NEXT_PUBLIC_APP_URL=.*|NEXT_PUBLIC_APP_URL=${URL}|" .env
sed -i "s|^OLLAMA_GPU=.*|OLLAMA_GPU=true|" .env
sed -i "s|^STT_FORCE_DEVICE=.*|STT_FORCE_DEVICE=cuda|" .env
# small model, NOT large-v3: the 3GB large-v3 download stalls silently on Kaggle
# (observed live -- port 3003 never came up, empty log). small is 460MB, loads in
# ~1 min, and on a T4 GPU transcribes a sentence in well under a second.
grep -q '^STT_MODEL=' .env && sed -i "s|^STT_MODEL=.*|STT_MODEL=small|" .env || echo 'STT_MODEL=small' >> .env
grep -q '^APP_INTERNAL_URL=' .env || echo 'APP_INTERNAL_URL=http://127.0.0.1:3000' >> .env
service postgresql start > /dev/null
PGPASS=$(grep -E '^PG_PASSWORD=' .env | cut -d= -f2 | awk '{print $1}')
sudo -u postgres psql -c "ALTER USER postgres PASSWORD '${PGPASS}';" > /dev/null
npm run db:setup && npm run db:check | tail -3

In [ ]:
%%bash
# ---- 10. Build + start everything (website, STT on GPU, voicebot) ----
cd /kaggle/working/app
npm run build 2>&1 | tail -3
nohup npm run start   > /kaggle/working/web.log      2>&1 &
cd server/stt-service
nohup python -u -m uvicorn app:app --host 127.0.0.1 --port 3003 > /kaggle/working/stt.log 2>&1 &
cd ..
nohup node voicebot-server.js > /kaggle/working/voicebot.log 2>&1 &
cd ..
# whisper small (~460MB) downloads + loads in about a minute — poll instead of a fixed sleep
for i in $(seq 1 24); do
  sleep 5
  READY=$(ss -tlnp | grep -cE ":(3000|3002|3003|11434) ")
  echo "[${i}0s] services up: $READY/4"
  if [ "$READY" = "4" ]; then break; fi
done
echo "--- listening ports (want 3000, 3002, 3003, 11434) ---"
ss -tlnp | grep -E ":(3000|3002|3003|11434)" | awk "{print \$4}"
echo "--- stt.log tail (if 3003 missing, this is why) ---"
tail -30 /kaggle/working/stt.log 2>&1
echo "--- warming up the Ollama model (else the FIRST call reply takes 30s+ and falls back) ---"
curl -s --max-time 180 http://127.0.0.1:11434/api/generate -d '{"model":"llama3.1:8b","prompt":"hi","stream":false}' -o /dev/null && echo "model warm and loaded on GPU"

In [ ]:
# ---- 11. Login cookie + ALL your URLs and next steps ----
import base64, hashlib, hmac, json, time, re

env = open("/kaggle/working/app/.env", encoding="utf-8").read()
secret = re.search(r"^AUTH_SECRET=(\S+)", env, re.M).group(1)
b64 = lambda b: base64.urlsafe_b64encode(b).rstrip(b"=").decode()
payload = b64(json.dumps({"email": "arjun996625@gmail.com", "role": "admin", "exp": int(time.time()) + 86400}, separators=(",", ":")).encode())
sig = b64(hmac.new(secret.encode(), payload.encode(), hashlib.sha256).digest())

web = open("/kaggle/working/tunnel_url.txt").read().strip()
wss = open("/kaggle/working/wss_url.txt").read().strip()
print("=" * 60)
print("1. OPEN THE WEBSITE :", web)
print("   Press F12 -> Console -> paste this line -> Enter:")
print(f'   document.cookie="rag_session={payload}.{sig}; path=/"')
print("   Then go to:", web + "/dashboard")
print("=" * 60)
print("2. EXOTEL SETUP (once per Kaggle session):")
print("   my.exotel.com -> App Bazaar -> your flow (1288523)")
print("   -> Voicebot applet -> set URL to:")
print("  ", wss)
print("   -> Save the flow.")
print("=" * 60)
print("3. Then run the last cell to place the real test call.")

# ---- publish the live URLs to GitHub (branch kaggle-live) so they are
# readable from outside while this kernel is still running (batch mode
# hides cell output until the run ends). Best-effort: a failure here must
# not kill the session -- the URLs above are still in this cell's output.
import subprocess
try:
    info = (f"web={web}\nwss={wss}\ncookie=rag_session={payload}.{sig}\n"
            f"dashboard={web}/dashboard\nstarted_utc={time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}\n")
    open("/kaggle/working/app/KAGGLE-LIVE.txt", "w").write(info)
    g = ["git", "-C", "/kaggle/working/app", "-c", "user.email=kaggle@rag.local", "-c", "user.name=kaggle-runner"]
    subprocess.run(g + ["add", "KAGGLE-LIVE.txt"], check=True)
    subprocess.run(g + ["commit", "-m", "kaggle live urls"], check=True, capture_output=True)
    subprocess.run(g + ["push", "--force", "origin", "HEAD:refs/heads/kaggle-live"], check=True, capture_output=True)
    print("\nLive URLs pushed to GitHub branch kaggle-live (KAGGLE-LIVE.txt)")
except Exception as e:
    print("\nWARNING: could not push live URLs to GitHub:", e)


In [ ]:
%%bash
# ---- 12. Health + GPU speed test (no phone call yet) ----
cd /kaggle/working/app
KEY=$(grep -E '^WHATSAPP_SERVICE_KEY=' .env | cut -d= -f2 | awk '{print $1}')
echo '--- TTS (expect edge-tts) ---'
curl -s -H "x-api-key: $KEY" http://127.0.0.1:3004/health
echo ''
echo '--- STT (expect model small, device cuda) ---'
curl -s -H "x-api-key: $KEY" http://127.0.0.1:3003/health
echo ''
echo '--- Priya brain speed on GPU (was 10-25s on the laptop) ---'
time curl -s -X POST http://127.0.0.1:3000/api/calls/turn -H 'Content-Type: application/json' \
  -H "x-api-key: $KEY" -d '{"event":"turn","callSid":"KAGGLE-TEST-1","speech":"hello, I want a home loan","language":"english"}'

In [ ]:
# ---- 13. REAL PHONE CALL — read before running! ----
# Priya will actually dial the number below. Requirements:
#   - Exotel account KYC/trial-verified for this number
#   - Step 2 from the URLs cell done (wss URL saved in the Exotel flow THIS session)
# Uses trial credits. Change CONFIRM to True, then run.

CONFIRM = False
PHONE   = "+919908838090"   # your verified test number

import re, urllib.request, json as j
if not CONFIRM:
    print("Not calling. Set CONFIRM = True (after doing step 2 in the URLs cell) and run again.")
else:
    env = open("/kaggle/working/app/.env", encoding="utf-8").read()
    import base64, hashlib, hmac, time
    secret = re.search(r"^AUTH_SECRET=(\S+)", env, re.M).group(1)
    b64 = lambda b: base64.urlsafe_b64encode(b).rstrip(b"=").decode()
    p = b64(j.dumps({"email": "arjun996625@gmail.com", "role": "admin", "exp": int(time.time()) + 3600}, separators=(",", ":")).encode())
    s = b64(hmac.new(secret.encode(), p.encode(), hashlib.sha256).digest())
    req = urllib.request.Request(
        "http://127.0.0.1:3000/api/calls",
        data=j.dumps({"phone": PHONE, "language": "telugu"}).encode(),
        headers={"Content-Type": "application/json", "Cookie": f"rag_session={p}.{s}"},
        method="POST",
    )
    try:
        print(urllib.request.urlopen(req, timeout=60).read().decode())
        print("\nPHONE SHOULD RING NOW. Answer it and talk to Priya!")
        print("Afterwards: check Voice Logs in the dashboard for the recording + transcript.")
    except urllib.error.HTTPError as e:
        print("Call failed:", e.read().decode())

In [ ]:
# ---- 14. KEEP THE WEBSITE RUNNING ----
# Batch/API-pushed kernels die the moment the last cell finishes, taking the
# website down with them. This cell just sleeps so everything stays up.
# Interactive users can stop it any time (services keep running without it).
# To end a batch session early: kaggle.com -> your notebook -> stop the run.
import time
HOURS = 8  # Kaggle GPU batch limit is ~9h; leave headroom
print(f"Website is LIVE. Keeping this session alive for up to {HOURS}h...")
for m in range(HOURS * 60):
    time.sleep(60)
    if m % 30 == 0:
        print(f"alive {m // 60}h{m % 60:02d}m -- stop the kernel on kaggle.com to end")
print("Keep-alive window over -- session will shut down now.")
